# **Automatic Mixed Precision**

**Autor**: Renan Santos Mendes

**Email**: renansantosmendes@gmail.com

## **O que é o uso de AMP com autocast**

O contexto `with torch.amp.autocast(device_type=...)` ativa o **Automatic Mixed Precision (AMP)** no PyTorch, permitindo que determinadas operações sejam executadas automaticamente em **menor precisão numérica** (como FP16 ou BF16), enquanto outras permanecem em FP32 para preservar a estabilidade numérica.

O objetivo é **acelerar o treinamento e reduzir o uso de memória**, explorando instruções especializadas do hardware (por exemplo, Tensor Cores em GPUs).

### **Como funciona**
- Operações matemáticas computacionalmente intensivas (ex.: matmul, convoluções) usam precisão reduzida
- Operações sensíveis à estabilidade numérica permanecem em FP32
- A escolha do tipo numérico é feita automaticamente pelo PyTorch

### **Benefícios**
- Maior throughput de treinamento
- Menor consumo de memória GPU
- Pouca ou nenhuma perda de qualidade do modelo

### **Uso típico**
- Treinamento de redes neurais profundas em GPU
- Modelos grandes (CNNs, Transformers, LSTMs)
- Cenários onde desempenho é crítico

O AMP não altera a arquitetura do modelo, apenas **como os cálculos são executados**, sendo transparente para o desenvolvedor.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import time

assert torch.cuda.is_available(), "Ative a GPU no Colab"

device = torch.device("cuda")

# ============================================================
# Modelo propositalmente pesado
# ============================================================

class HeavyMLP(nn.Module):
    def __init__(self, input_dim=1024, hidden_dim=2048, depth=6):
        super().__init__()
        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.ReLU())

        for _ in range(depth):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.ReLU())

        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model_fp32 = HeavyMLP().to(device)
model_amp = HeavyMLP().to(device)

# ============================================================
# Dados grandes (para saturar a GPU)
# ============================================================

batch_size = 256
x = torch.randn(batch_size, 1024, device=device)
y = torch.randn(batch_size, 1, device=device)

criterion = nn.MSELoss()

# ============================================================
# Função de benchmark
# ============================================================

def train_step(model, use_amp=False):
    optimizer = optim.AdamW(model.parameters(), lr=1e-3)
    scaler = torch.amp.GradScaler(enabled=use_amp)

    model.train()

    start = time.time()

    for _ in range(50):
        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            with torch.amp.autocast(device_type="cuda"):
                preds = model(x)
                loss = criterion(preds, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            preds = model(x)
            loss = criterion(preds, y)
            loss.backward()
            optimizer.step()

    torch.cuda.synchronize()
    return time.time() - start


# ============================================================
# Benchmark
# ============================================================

time_fp32 = train_step(model_fp32, use_amp=False)
time_amp = train_step(model_amp, use_amp=True)

print(f"FP32 time: {time_fp32:.2f}s")
print(f"AMP  time: {time_amp:.2f}s")
print(f"Speedup: {time_fp32 / time_amp:.2f}x")

# ============================================================
# Memória
# ============================================================

print("\nGPU memory usage:")
print(f"FP32 allocated: {torch.cuda.max_memory_allocated() / 1024**2:.2f} MB")

torch.cuda.reset_peak_memory_stats()
train_step(model_amp, use_amp=True)
print(f"AMP allocated: {torch.cuda.max_memory_allocated() / 1024**2:.2f} MB")


## O que significa ONNX?

**ONNX** é a sigla para **Open Neural Network Exchange**.

Trata-se de um **formato aberto e padronizado** para representar modelos de aprendizado de máquina e deep learning, de forma **independente de framework**.

Em termos simples:

> **ONNX é uma “língua franca” para modelos de IA.**

---

## Para que o ONNX serve?

O ONNX resolve um problema central em Machine Learning moderno:

> **treinar em um framework e executar em outro ambiente, com máxima eficiência.**

Ele permite **separar claramente duas fases** do ciclo de vida de um modelo:

| Fase | Ferramenta típica |
|----|----------------|
| Treinamento | PyTorch, TensorFlow, JAX |
| Inferência (produção) | ONNX Runtime, TensorRT, OpenVINO |

---

### Integração com aceleradores

ONNX Runtime suporta:
- CPUExecutionProvider
- CUDAExecutionProvider
- TensorRTExecutionProvider
- OpenVINOExecutionProvider

---

## Relação entre PyTorch e ONNX

### PyTorch
- Excelente para:
  - Pesquisa
  - Treinamento
  - Experimentação
- Menos ideal para:
  - Inferência em larga escala
  - Ambientes restritos

### ONNX
- Excelente para:
  - Inferência
  - Deploy
  - Performance
- Não serve para:
  - Treinar modelos


In [ ]:
%%capture
!pip install -q onnx onnxruntime onnxscript

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, input_dim=100, hidden_dim=128, output_dim=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
torch.manual_seed(42)

X = torch.randn(5000, 100)
y = torch.randint(0, 10, (5000,))


In [ ]:
model = SimpleMLP()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

model.train()
for epoch in range(10):
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()

In [ ]:
model.eval()

input_sample = torch.randn(1, 100)

with torch.no_grad():
    start = time.time()
    for _ in range(1000):
        _ = model(input_sample)
    torch_time = time.time() - start

print(f"Tempo PyTorch (1000 inferências): {torch_time:.4f} s")


In [ ]:
onnx_path = "simple_mlp_2.onnx"

torch.onnx.export(
    model,
    input_sample,
    onnx_path,
    export_params=True,
    opset_version=17,
    input_names=["input"],
    output_names=["output"],
    dynamic_shapes={"x": {0: torch.export.Dim("batch_size")}}
)

print("Modelo exportado para ONNX")


In [ ]:
import onnxruntime as ort

In [ ]:
session = ort.InferenceSession(
    onnx_path,
    providers=["CPUExecutionProvider"]
)

In [ ]:
input_name = session.get_inputs()[0].name
input_numpy = input_sample.numpy()

In [ ]:
start = time.time()
for _ in range(1000):
    _ = session.run(None, {input_name: input_numpy})
onnx_time = time.time() - start

print(f"Tempo ONNX Runtime (1000 inferências): {onnx_time:.4f} s")

In [ ]:
speedup = torch_time / onnx_time

print("\nResumo:")
print(f"PyTorch      : {torch_time:.4f} s")
print(f"ONNX Runtime : {onnx_time:.4f} s")
print(f"Speedup      : {speedup:.2f}x")